## Sustainability of AI - Research Assistant

### Objective
Develop a RAG enabled chatbot with a Gradio interface that answers questions using a curated collection of research articles on the environmental impacts of AI. 

### RAG Flow
- Load Markdown versions of research articles
- Chunk the articles 
- Retrieve the most relevant chunks based on the question
- Add those chunks to the model context 
- Generate an answer with source citations
- Expose this workflow through a Gradio interface

In [1]:
#Setup environment variables
import os
import sys
from pathlib import Path
from collections import Counter
import numpy as np
import gradio as gr

# Make the project root importable (notebook is run from the notebooks/ folder)
sys.path.append(str(Path.cwd().parent))

from src.sustainability_rag.config import DATA_DIR, TOP_K, EMBEDDING_MODEL_NAME, PROVIDER, MODELS
from src.sustainability_rag.documents import load_documents, build_source_registry, chunk_by_section
from src.sustainability_rag.embeddings import get_embedding_model, generate_embeddings, embed_chunks
from src.sustainability_rag.retrieval import retrieve_relevant_chunks, build_context
from src.sustainability_rag.rag_pipeline import (
    SYSTEM_PROMPT, answer_question, clients, MODEL, client,
    openai_api_key, anthropic_api_key, openrouter_api_key,
)

assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"

In [2]:
#Validate API Keys (keys are loaded from .env in src/sustainability_rag/rag_pipeline.py)
if openai_api_key:
    print(f"OpenAI API Key configured: {bool(openai_api_key)}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key configured: {bool(anthropic_api_key)}")
else:
    print("Anthropic API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key configured: {bool(openrouter_api_key)}")
else:
    print("OpenRouter API Key not set")

OpenAI API Key configured: True
Anthropic API Key configured: True
OpenRouter API Key configured: True


In [3]:
# Client Setup and active model configuration live in src/sustainability_rag/
# (change PROVIDER in config.py, then restart the kernel)
print(f"Using {PROVIDER} with model {MODEL}")


Using openai with model gpt-4.1-mini


In [4]:
docs = load_documents()
source_registry = build_source_registry(docs)
len(docs), docs[0]["source"], docs[0]["content"][:200]  # Display the number of documents, the first document's source, and a preview of its content

(8,
 'alpine_2026_AI_indirect_emissions.md',
 '# AI-driven productivity gains enable more CO2 emissions than they avoid in a global energy-economy model\n\nAuthors: Will Alpine, Nathan Geldner, Holly Alpine, Maksym G. Chepeliev\nPublication Year: 202')

In [5]:
all_chunks = []
for doc in docs:
    all_chunks.extend(chunk_by_section(doc, source_registry))

content_chunks = [chunk for chunk in all_chunks if chunk["type"] == "content"]

print(f"Total number of chunks created: {len(all_chunks)}")
print(f"Number of content chunks: {len(content_chunks)}")
print(content_chunks[0])

Counter(chunk["type"] for chunk in all_chunks)

Total number of chunks created: 267
Number of content chunks: 253
{'source_id': 'S1', 'source': 'alpine_2026_AI_indirect_emissions.md', 'section': '### Background: fossil fuels and the energy transition', 'text': "Climate change presents profound societal and economic risks, and limiting global warming to 1.5 C requires rapid decarbonization centered on phasing out fossil fuels. Yet fossil fuels continue to supply over 80% of global primary energy and global consumption continues to rise, despite renewables' increasing deployment and cost parity. This reflects energy addition rather than substitution: renewables have historically expanded total energy supply while fossil fuels remain largely undisplaced. Beyond electricity, fossil fuels also remain deeply embedded in industrial pathways throughout the broader global economy. Path-dependent co-evolution of technological systems with governing institutions reinforces incumbent energy regimes. Together, these dynamics complicate efforts t

Counter({'content': 253, 'executive_summary': 8, 'abstract': 6})

In [6]:
embeddings = embed_chunks(content_chunks)
print(f"Embeddings generated for {len(embeddings)} chunks. Each embedding has a shape of {embeddings[0].shape}.")

Embeddings generated for 253 chunks. Each embedding has a shape of (384,).


In [7]:
retrieved_chunks = retrieve_relevant_chunks(
    query="What are the environmental impacts associated with model training?",
    content_chunks=content_chunks,
    embeddings=embeddings,
    top_k=TOP_K
)

retrieved_chunks[:3]  # Display the top 3 retrieved chunks based on the query

[({'source_id': 'S3',
   'source': 'luccioni_2024_power_hungry_processing.md',
   'section': '### Inference studies',
   'text': "While most work has focused on training because it is more tractable (usually carried out over a set period on a specific compute instance), model inference has started to become the subject of scholarship. Luccioni et al.'s study of BLOOM was the first to look at the specific energy costs of deploying an LLM and found that, over time, this can represent a significant portion of a model's overall carbon footprint. The current study pursues this line of work, testing a variety of architectures on different tasks and datasets to understand the degree of variance and how seemingly small user choices can result in large differences in a model's environmental impact.",
   'type': 'content'},
  np.float32(0.6332047)),
 ({'source_id': 'S2',
   'source': 'luccioni_2021_carbon_footprint_of_bloom.md',
   'section': '### Empirical Studies on ML CO2 Emissions',
   'text

In [8]:
test_questions = [
    "How much water does AI consume?",
    "What are rebound effects?",
    "What assumptions drive carbon accounting studies?",
    "Are training emissions larger than inference emissions?",
    "What limitations exist in carbon accounting studies?"
]

question = test_questions[0]

results = retrieve_relevant_chunks(
    query=question,
    content_chunks=content_chunks,
    embeddings=embeddings,
    top_k=TOP_K
)

print(f"QUERY: {question}")

for rank, (chunk, score) in enumerate(results, start = 1):
    print("=" * 80) # Separator for readability
    print(f"Rank: {rank}")  # Display the rank of the chunk
    print(f"Score: {float(score):.4f}")  # Display the similarity score
    print(f"Source: {chunk['source']}")  # Display the source of the chunk
    print(f"Section: {chunk['section']}")  # Display the section of the chunk
    print(chunk["text"][:500])  # Display the first 500 characters of each retrieved chunk for brevity
    print()  # Add a newline for better readability between chunks

QUERY: How much water does AI consume?
Rank: 1
Score: 0.5797
Source: watershed_ghg_emissions_ai_framework.md
Section: ### Broader context
AI emissions accounting is part of broader public and policy concerns about data center growth, which a defensible framework must acknowledge even where they fall outside the strict GHG inventory.
- Local air quality: AI infrastructure concentrates load geographically, which can produce criteria pollutant exposure (NOx, PM2.5, SO2), particularly where backup generation runs on natural gas or diesel. Carbon-only accounting misses these distributional impacts.
- Water use: AI workloads can be wate

Rank: 2
Score: 0.5597
Source: xiao_net_zero_pathways_ai_servers.md
Section: ### Efficiency innovations, rebound effects and recommendations
Emerging high-efficiency technologies in hardware and software, exemplified by DeepSeek, may transform AI server supply and demand. Across the study's five scenarios, they may cause deviations of up to 393 million m3 in 

In [9]:
context = build_context(results)
print(len(results))
print(f"CONTEXT:\n{context[:3000]}")  # Display the first 3000 characters of the context for brevity

5
CONTEXT:
[S7]Source: watershed_ghg_emissions_ai_framework.md
Section: ### Broader context
Text: AI emissions accounting is part of broader public and policy concerns about data center growth, which a defensible framework must acknowledge even where they fall outside the strict GHG inventory.
- Local air quality: AI infrastructure concentrates load geographically, which can produce criteria pollutant exposure (NOx, PM2.5, SO2), particularly where backup generation runs on natural gas or diesel. Carbon-only accounting misses these distributional impacts.
- Water use: AI workloads can be water-intensive at the cooling stage or entail withdrawals and consumption for electricity generation. The WUE row in Table 5 is a starting point tracking on-site water use. Electricity figures could be used to calculate upstream (off-site) water use from power generation, but this is less tractable than the carbon analogue: (1) water accounting requires distinguishing withdrawal (diverted and largely r

In [10]:
#Preflight tests before LLM call

assert len(docs) > 0, "No documents loaded. Please check the data directory."
assert len(content_chunks) > 0, "No content chunks created. Please check the chunking logic."
assert len(embeddings) == len(content_chunks), "Mismatch between number of embeddings and content chunks."
assert np.isfinite(embeddings).all(), "Embeddings contain non-finite values. Please check the embedding generation process."
assert len(question) > 0, "The question cannot be empty. Please provide a valid question."

print("All preflight tests passed. Ready to answer questions.")

preflight_results = retrieve_relevant_chunks(
    query="How much water does AI consume?",
    content_chunks=content_chunks,
    embeddings=embeddings,
    top_k=TOP_K
)

assert len(preflight_results) > 0, "No relevant chunks retrieved. Please check the retrieval logic."

for chunk, score in preflight_results:
    assert chunk["text"], "Retrieved chunk has empty text. Please check the chunking and retrieval logic."
    assert chunk["source"], "Retrieved chunk has empty source. Please check the chunking and retrieval logic."
    assert -1.0 <= score <= 1.0, f"Similarity score {score} is out of expected range [-1, 1]. Please check the similarity computation."

print("All preflight retrieval tests passed. Retrieval logic is functioning as expected.")

preflight_context = build_context(preflight_results)
print(f"Preflight context length: {len(preflight_context.split())} words.")

All preflight tests passed. Ready to answer questions.
All preflight retrieval tests passed. Retrieval logic is functioning as expected.
Preflight context length: 1093 words.


In [11]:
response = answer_question("What are rebound effects?", content_chunks, embeddings)

for i, (chunk, score) in enumerate(response["retrieved_chunks"], start=1):
    print(f"Chunk {i}:")
    print(f"Source: {chunk['source']}")
    print(f"Section: {chunk['section']}")
    print(f"Score: {score:.4f}")
    print(f"Text Preview: {chunk['text'][:200]}...")
    print("-" * 40)

print(response["answer"])
print("\nSources Used:")
print("-" * 40)
for source_id, info in response["source_map"].items():
    print(f"{source_id}: {info['source']} (Section: {info['section']})")

Chunk 1:
Source: alpine_2026_AI_indirect_emissions.md
Section: ### Limitations, excluded dynamics, and conservative assumptions
Score: 0.2914
Text Preview: Figure 1 illustrates the analytical boundary of the model, distinguishing modeled, partially modeled, and unmodeled mechanisms (extended discussion in SI Notes S4 and S5).

**Structural constraints.**...
----------------------------------------
Chunk 2:
Source: oviedo_2026_energy_use_of_ai_inference.md
Section: ### Rebound effect
Score: 0.2737
Text Preview: Efficiency gains do not guarantee lower absolute energy use. Lower energy per query, lower cost per token, or increases in model size can enable higher usage, longer generations and more token-intensi...
----------------------------------------
Chunk 3:
Source: alpine_2026_AI_indirect_emissions.md
Section: ### Impact taxonomy
Score: 0.2538
Text Preview: Following the European Green Digital Coalition (EGDC) taxonomy, the paper distinguishes:
- First-order effects: direct operation